# 数据处理

下载和读取， 参考：<https://github.com/karpathy/nanoGPT/blob/master/data/shakespeare/prepare.py>

```python
import os
import requests

# os.getcwd()    会放到 code文件夹下面，而不是 zero_gpt文件夹下面
# __file__(当前文件路径)，会放到和当前文件同一个文件夹下，但是jupyter里不能用
input_file_path = os.path.join(os.path.dirname(os.getcwd()), 'input.txt')
if not os.path.exists(input_file_path):
    data_url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
    with open(input_file_path, 'w', encoding='utf-8') as f:
        f.write(requests.get(data_url).text)

# 网络不好的话还是自己手动下载放好吧
```

## 读取数据

In [1]:
text = open("input.txt", 'r').read()
print("length of datasets in characters: ", len(text))
print(text[:100])

length of datasets in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


## 分词器构建

In [2]:
# 统计字典的字符数量
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
char_ascii_num = [ord(char) for char in chars]  # chr()和ord() ord() 把字符转成数字编码，chr() 把数字编码转回字符‌
print(char_ascii_num)
# 第一个字符应该是Line Feed（直译为“送纸”或“行推进”）。
# Line Feed (LF)，将光标移动到下一行的相同水平位置（在现代计算机中通常默认也会回到行首）。
# Carriage Return (CR, 回车，ASCII 13)
# 10是换行  32是space空格 33是！
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
[10, 32, 33, 36, 38, 39, 44, 45, 46, 51, 58, 59, 63, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122]
65


In [3]:
# 字符到数字的映射， 和数字到字符的映射
# 最简单的分词 tokenizer过程
# 这里和以前不一样了，以前会加一个 . 作为终止符
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for ch,i in stoi.items()}

encode = lambda s: [stoi[c] for c in s]  # lambda函数， s表示sentence/string 把一串字符编码为整数列表
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hello world!"))
print(decode(encode("hello world!")))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42, 2]
hello world!


还有很多分词方法，比如：
+ [openai/tiktoken](https://github.com/openai/tiktoken)
+ [google/sentencepiece](https://github.com/google/sentencepiece)

In [4]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
print(enc.n_vocab)

print(enc.encode("hello world!"))
print(enc.decode([31373, 995, 0]))

50257
[31373, 995, 0]
hello world!


## 构造数据集

In [5]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)  # torch.long，即 int64
print(data.shape, data.dtype)
print(data[:20])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56])


In [6]:
# 划分训练集和验证集
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

In [7]:
# 限制最大输入长度，因为不可能一下子把 1115394 这整个文档的char全都送到网络里去
block_size = 8
train_data[: block_size+1] 
# 这里加1表示的是用前8个字符预测第9个字符，即：在构造的数据集样本中，输入是前8个字符，输出是要预测的下一个字符
# 由于滑动窗口的存在，这里看似是9个字符，实际上包含了8个样本，全为空不算，所以可以理解为 7个空+第一个字符→第二个字符，...  0个空+8个字符→最后一个字符
# 其实快速判断的方法就是看 输出有多少种样本，很明显，除了第一个字符之外，其余8个都可以作为输出，所以有8个样本

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [8]:
demo_x = train_data[: block_size]
demo_y = train_data[1: block_size+1]
for t in range(block_size):
    context = demo_x[:t+1]  # t从0开始 [:1] 相当于取0 
    target = demo_y[t]
    print(f"when input is [{context}], the target is [{target}]")

when input is [tensor([18])], the target is [47]
when input is [tensor([18, 47])], the target is [56]
when input is [tensor([18, 47, 56])], the target is [57]
when input is [tensor([18, 47, 56, 57])], the target is [58]
when input is [tensor([18, 47, 56, 57, 58])], the target is [1]
when input is [tensor([18, 47, 56, 57, 58,  1])], the target is [15]
when input is [tensor([18, 47, 56, 57, 58,  1, 15])], the target is [47]
when input is [tensor([18, 47, 56, 57, 58,  1, 15, 47])], the target is [58]


In [9]:
a = torch.randint(300, (8,))
a

tensor([143,  28,  55, 201, 147, 232, 187,  74])

In [10]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split, batch_size = 4):
    """
    split：数据集划分，例如：train_data/val_data
    """
    data = train_data if split=="train" else val_data
    ix = torch.randint(len(data)-block_size, (batch_size,))  # low=0, high, size(tuple) 即：在0~1115394-8 这堆数里，生成4个索引值 这4个值是随机抽的，不是连续的，和data的连续的数字映射无关
    x = torch.stack([data[i:i+block_size] for i in ix]) # 默认dim = 0，堆叠成多行
    y = torch.stack([data[i+1:i+block_size+1] for i in ix]) 
    # y等于x向右偏移1个，所以加1就行，这里和之前的makemore不同，之前是输入3个，输出1个；现在是输入3个，输出3个(输入的后2个+预测的1个，这里输入不要第一个)
    # 这个构建方法就和大模型/基于Transformer的预测是一致的了
    return x,y

xb,yb = get_batch('train')
print(f"inputs: {xb}\n{xb.shape}")
print(f"targets: {yb}\n{yb.shape}")
print("-----")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is [{context}], the target is [{target}]")

inputs: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
torch.Size([4, 8])
targets: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
torch.Size([4, 8])
-----
when input is [tensor([24])], the target is [43]
when input is [tensor([24, 43])], the target is [58]
when input is [tensor([24, 43, 58])], the target is [5]
when input is [tensor([24, 43, 58,  5])], the target is [57]
when input is [tensor([24, 43, 58,  5, 57])], the target is [1]
when input is [tensor([24, 43, 58,  5, 57,  1])], the target is [46]
when input is [tensor([24, 43, 58,  5, 57,  1, 46])], the target is [43]
when input is [tensor([24, 43, 58,  5, 57,  1, 46, 43])], the target is [39]
when input is [tensor([44])], the target is [53]
when input is [tensor([44, 53])], the target is [5

根据[大模型实战营第二期——4. XTuner 大模型单卡低成本微调实战->1.1 增量预训练微调](https://stitch.blog.csdn.net/article/details/136100537)

```bash
data: <s>世界第一高峰是珠穆朗玛峰<s>
label: 世界第一高峰是珠穆朗玛峰<s>
# 是用输出+停止符 作为输出，即：损失计算的对象
# 输入的首字符/起始字符，是不作为输出/计算损失的对象的

# 计算损失函数的时候，就相当于
# 输入，      预测输出，
  <s> 预测下一个 世
  <s>世  预测下一个 界
  ...
# 会逐个匹配的，所以输入比输出多一个起始字符

# 但是这里直接构造的输入和输出是一样长，只要保证输入 能够对到 输出 是预测下一个即可
```

这32个独立样本被打包作为一个batch送到网络中，所以看似是4个输入长度为8的序列，但是实际构成的样本数有32个。。。

# 模型

从最简单的模型开始，
1. bigram模型 → `2_build_makemore.ipynb`里 使用词频计数作为预测的方案

In [11]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

## Bigram model

这其实就是重新实现了`2_build_makemore.ipynb`里 使用词频计数作为预测的方案，用这个词频表作为决定下一个词是谁的方案，输入是单个字符，输出也是单个字符

In [12]:
vocab_size

65

### 模型定义

In [13]:
# Module 是类，首字母大写。
# modules 不是给你继承用的类，它通常是一个包/模块命名空间。
# 不要写成 class BigramLanguageModel(nn.modules):
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # nn.Embedding(num_embeddings: int,embedding_dim: int)
        # 每个token直接从这个查询表里读取下一个token的logits（即：vocab_size个元素的逻辑值，进而计算vocab_size个元素的概率，并从概率中采样下一个字符的index）
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets= None):
        """
        这里将targets设置为可选，是为了 generate中self.forward(idx)能够正常运行
        """
        # idx和targets都是(B,T)维度的整数 int
        # 即 (batch, timestamp), (批量大小,序列长度)
        # 所以logits的shape为: (B,T,C), 这里的C就是 vocab_size，老师的解释是 channel，即：每个字符的表示维度刚好是vocab_size，即num_feature/channel
        logits = self.token_embedding_table(idx)
        # loss = F.cross_entropy(logits, targets) 
        # logits.shape(4,8,65) yb.shape(4,8) 不符合 F.cross_entropy接受的参数维度，所以需要改成下面这样
        if targets is None:
            loss = None
        else:
            loss = F.cross_entropy(logits.view(-1,vocab_size), targets.view(-1))
            # 老师是这么写的，很清晰，但是很麻烦
            # B,T,C = logits.shape
            # logits = logits.view(B*T,C)
            # targets = targets.view(B*T)
        return logits,loss
        
    def generate(self, idx, max_new_token):
        """
        idx: (B,T) array of indices in the current context
        这个其实和 `3_MLP_makemore.ipynb# 采样` 的实现是一样的
        """
        for _ in range(max_new_token):
            logits,loss = self.forward(idx) # logits.shape(4,8,65)
            # 注意，这里idx是(B,T)形状，例如:(4,8)就可以理解为 4个样本，每个样本的上下文长度为8
            # 而这样输入带来的输出也是(4,8,65),即4个样本，上下文长度为8，每个上下文的下一个预测结果
            # 但是我们只关注最后一个，即: 从(4,8,65)→(4,1,65)
            logits = logits[:,-1,:] 
            probs = F.softmax(logits,dim = -1) # 对 65这个维度求和计算e^x
            # 这里和以前不一样， 以前是replacement=True，这里默认不改的话是False
            idx_next = torch.multinomial(probs, num_samples = 1)
            idx = torch.cat((idx, idx_next), dim = 1) # (B, T+1)
        return idx

目前的写法看起来有点傻，明明只需要输入前一个字符就可以预测下一个，但是这里`generate()`函数里输入了8个去预测下一个

这是因为目前是个简单的 bigram，用到的上下文只有1个字符；但是如果换成后面真正的Transformer或者更复杂的模型，这样的输入就合理了

这样做是为了保证函数的一致性

In [14]:
# m 即 model
m = BigramLanguageModel(vocab_size) 
out, loss = m(xb,yb)
print(out.shape, loss) 

demo_generate = m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=100)[0].tolist()
# 拆分一下就是：
# idx = torch.zeros((1,1), dtype=torch.long)  # batch为1，上下文长度也为1，值为0 即：换行符
# generate_result = m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=100) # 生成100个新字符
# show_one = generate_result[0].tolist() #转成列表
print(decode(demo_generate)) # 数字 映射 为对应的 字符

torch.Size([4, 8, 65]) tensor(4.8786, grad_fn=<NllLossBackward0>)

Sr?qP-QWktXoL&jLDJgOLVz'RIoDqHdhsV&vLLxatjscMpwLERSPyao.qfzs$Ys$zF-w,;eEkzxjgCKFChs!iWW.ObzDnxA Ms$3


之前在  **4_MLP2_makemore.ipynb## 初始化分析**中，有计算过损失的标准值
+ 这里的词表大小是65，假设每个类别/单词 都是相等概率取到，则
+ 根据损失函数的公式： $loss = -\frac{1}{N}\sum_{i=1}^Nlog(y_{i})$
+ 则这里就是
  ```python
  ```

In [15]:
import math
-(math.log(1/65))*32/32  # 4.174387269895637
# 得到的结果和上面的初始loss = 4.6630 有一定的差距，说明上面的网络初始预测的分布并不均匀

4.174387269895637

In [16]:
yb.shape
# logits.shape 明显是 (4,8,65)
# 所以这里需要把 对维度进行转换

torch.Size([4, 8])

(B,T,C)维度， `B(batch size) = 4`批量大小是4, `T(timestamp) = 8`序列长度是8，`C(vocab_size) = 65`表示词表大小
+ 所以其实是对`4*8=32`个`timestamp`都预测下一个词的index
+ 而不是单纯预测`4`个样本的第8个字符后的下一个词的index

这个模型的特征就是：
+ 每个字符只能看到自己，看不到任何其他的上下文，即：输入的4个样本的8个字符是没有上下文的，都是单个字符直接去预测下一个~

----

这里由于 `F.cross_entropy`对输入和目标的维度规定，所以无法直接`loss = F.cross_entropy(idx, targets)`， 根据[torch.nn.functional.cross_entropy](https://docs.pytorch.org/docs/2.13/generated/torch.nn.functional.cross_entropy.html)
```bash
C = number of classes
N = batch size
input: shape(C),(N,C),(N,C,d1,d2,...,dk), 其中如果是K维loss，K>=1
Target: 如果包含indices，shape(),(N),(N,d1,d2,...,dk)

所以可以发现，F.cross_entropy的目标不能接受 one-hot，必须是一个样本一个类别值这样精确的标签
```

In [17]:
m.token_embedding_table.weight.shape, xb.shape

(torch.Size([65, 65]), torch.Size([4, 8]))

### 训练和评测

In [18]:
m = BigramLanguageModel(vocab_size) 

In [19]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)
# 一般这个优化器的学习率是 3e-4, 但是这里因为网络比较小，很简单，所以可以直接 1e-3

batch_size = 32
for steps in range(10000):
    # 这里优化的步数，可以随便设置，大概 14k能到 2.399 这个loss结果
    xb,yb = get_batch('train', batch_size)
    logits, loss = m(xb,yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()

    # 以前的做法  3_MLP_makemore.ipynb# 开始优化（5个单词） 里写的
    # for p in parameters:
    #     p.grad = None
    # 等价于 optimizer.zero_grad(set_to_none = True)
    # loss.backward()
    # for p in parameters:
    #     p.data-=lr*p.grad
    # 等价于  optimizer.step()
print(loss.item())

2.4505205154418945


对比：`2_build_makemore.ipynb## 评测模型效果/性能` 这里得到的损失是 `2.4241`（但是这里是27个字符的词表, 预测下一个字符的交叉熵损失应该是从 -math.log(1/27) = 3.295836866004329）

这里是从 `-(math.log(1/65)) = 4.174387269895637`优化到的`2.399`, 优化的差异还是存在的

In [20]:
-math.log(1/27)

3.295836866004329

In [21]:
print(decode(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=400)[0].tolist()))


CI
TEE:
NClucor che t thendouilftheesco'imin?
-mapp;
Dupll nn owe hisaloriclfer eened de ms, sth ht hon tearow, att t isin aren gs fran bag ndais theer so,

ITh mounce s pr higay'stindinthe,

Why w'surnurn myof bby fedise?
Thertle:
LOLO, e te.
Fzern hithe l ath
an w, KI ardathart.
3D m arere toor torgo sh meas hirend


As anicrirtoprth ovend sinouce doreak anotoreranmye, bithorer glre?
ARSLis i'Ch


比起上面直接初始化推理生成的结果，要好多了。。。至少看起来不是一群乱码了，有空格了

**这是从回车符开始预测的结果**， 已经很6了

```bash
,'e&E
kvrRWjEiNwc;mX'ASnJnijArEzlNLHkklqdnopvjO.Ye!RahUADiN.Ve,;3YY3IPzaPqSFnM nFkpeRA3FmqiI;l, HE
y
```

### 总结

Bigram这种模型，token之间没有关系，只有当前词和下一个词，其余没有什么上下文，窗口太窄

接下来的Transformer需要token之间开始交流（即：token之间可以看到上下文了, 窗口变大~）

## Transformer

### 自注意力机制中的数学技巧

**the mathematical trick in self attention**

In [22]:
# 简单示例
import torch
torch.manual_seed(1337)
B,T,C = 4,8,2   # batch timastap channel
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

关于自注意力机制，举例说明：
+ 就是这里4个样本，每个样本的8个token，我们希望这8个token，彼此之间可以关联起来，可以计算相关性

这里的规则是，
+ 5th的token只能和1st,2nd,3rd,4th这前面的几个token，或者说过去/历史token关联，
+ 不可以和6th,7th,8th关联，因为这是要预测的未来
+ 注意，**这里说的是自注意力机制的构建，相当于只对输入进行处理，不是损失函数，所以看不到输出**

不难想到：
+ 最简单的计算当前token和之前token关系的方式就是计算平均值
+ 例如:
  + 对第一个token来说，关系值 = 本身
  + 对第二个token来说，关系值 = (第一个token + 本身)/2
  + ...
  + 对第$n$个token来说， 关系值 = (第一个token + 第二个token + ... + 第n个token本身)/n
+ 这种方式**会损失历史信息的时序关系**，但是很简单

In [23]:
%%timeit
# %%time
# x[b,t] = mean_{i≤t}(x[b,i])
# 好像没什么快捷方式可以实现，直接for循环了
# bow: bag of word 词袋模型 这里称为词袋，是因为这里的关系和词袋一样简单，原始的词袋装的是频数/出现次数，这里装的是平均数
# 都是很好算的统计特征
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]  # 得到的结果是 (t, C)
        xbow[b,t] = torch.mean(xprev, dim = 0) # 在t这个维度进行平均 t这个维度消失

3.48 ms ± 201 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


`%%timeit`好像会运行完清空那部分变量，所以要重新单独跑一遍，不然回提示没有`xbow`这个变量

In [26]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]  # 得到的结果是 (t, C)
        xbow[b,t] = torch.mean(xprev, dim = 0) # 在t这个维度进行平均 t这个维度消失

In [27]:
sample_num = 0    # 这里是batch=4的一批里的第几个， C=2
# 第0个的关系和第0个的通道/特征值 是一样的
# 后面才开始体现平均
print(x[sample_num])
print(xbow[sample_num])

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


上面这种方式虽然可以达到目的，但是计算效率很低，完全可以用矩阵乘法来高效的实现上面的结果

### 我的求均值改为矩阵乘法

理解多维矩阵乘法维度变化的核心法则只有两句话：
+ **最后两个维度**：执行标准的二维矩阵乘法规则（即 `(m, n) @ (n, p) -> (m, p)`）。
+ **前面的所有维度**：视为“批量（Batch）”维度，必须完全相同或满足广播（Broadcasting）机制。

所以上面的例子： `(4,8,2)@? = (4,8,2) `
+ 由于这里计算是在8上面发生的，所以最好8是放在列上
+ 所以应该就是 `(4,8,8)@(4,8,2) = (4,8,2)`

同时这里的`(4,8,8)`的矩阵， 先考虑基本的计算单元：`(8,8)@(8,2)`的部分，

`(8,8)`矩阵应该类似下面这样
```bash
1, 0, 0, 0, 0, 0, 0, 0              # 1个1,7个0
1/2, 1/2, 0, 0, 0, 0, 0, 0          # 2个1/2,6个0
1/3, 1/3, 1/3, 0, 0, 0, 0, 0        # 3个1/3,5个0
...
...
1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8, 1/8  # 8个1/8
```
这样就符合第一个关系值是第一个token自己，第二个关系值是前两个token的均值，第三个关系值是前三个token的均值,...这样的规律了

In [47]:
%%timeit
# %%time
# 我自己初步的结果

# ================= 1. 构建 8x8 基础矩阵 =================
# 生成 8x8 的下三角矩阵（对角线及以下为1，以上为0）
n = 8
mask = torch.tril(torch.ones(n, n))

# 生成每行的除数 [1, 2, ..., 8]，并增加一个维度变成列向量 (8, 1)
divisors = torch.arange(1, n + 1, dtype=torch.float32).unsqueeze(1)

# 利用广播机制进行按行除法：(8, 8) / (8, 1) -> (8, 8)
matrix_88 = mask / divisors

# ================= 2. 复制 4 份得到 (4, 8, 8) 张量 =================
# 先在头部增加一个维度，将 (8, 8) 变为 (1, 8, 8)
matrix_188 = matrix_88.unsqueeze(0)

# 逻辑复制 (共享内存)
# tensor_4x8x8 = matrix_1x8x8.expand(4, -1, -1)

# 物理复制 (分配新内存)
tensor_488 = matrix_188.repeat(4, 1, 1)

relation_x = tensor_488@x

387 μs ± 58.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [30]:
n = 8
mask = torch.tril(torch.ones(n, n))
divisors = torch.arange(1, n + 1, dtype=torch.float32).unsqueeze(1)
matrix_88 = mask / divisors
matrix_188 = matrix_88.unsqueeze(0)
tensor_488 = matrix_188.repeat(4, 1, 1)
relation_x = tensor_488@x

In [31]:
# print( (xbow[0] == relation_x[0]).all())  
# 不能这么写，打印出来值一样，但是这样判断就不相等。。
# 使用 == 运算符时，PyTorch 进行的是绝对相等（Bitwise equality） 的比较。只要底层二进制表示有 1 bit 的不同，就会返回 False。
# 所以直接矩阵计算和for循环的浮点数计算还是有差异的
print(torch.allclose(xbow[0], relation_x[0]))
print(xbow[0])
print(relation_x[0])

True
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


### ✅标准的求均值改为矩阵乘法（为什么B批次在第一个维度）

这里给个简单的例子

In [38]:
torch.manual_seed(42)
# a = torch.ones(3,3)
a = torch.tril(torch.ones(3,3))  # 只需要对a进行归一化，就可以求平均值了
a = a/torch.sum(a, dim = 1,keepdim=True)  # 不写keepdim=True就得不到正确结果
b = torch.randint(0, 10, (3,2)).float()
c = a@b
print(f"a = \n{a}")
print(f"b = \n{b}")
print(f"c = \n{c}")

a = 
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b = 
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
c = 
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [33]:
torch.tril(torch.ones(3,3))

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

由此可以知道，向量化的实现方案应该是：

In [43]:
print(x.shape)   # B,T,C = 4,8,2
# 这里是 a@b  即：下三角矩阵@x           (4,8,8)@(4,8,2)=(4,8,2)
# 所以和我上面推理的结果类似，这个下三角矩阵是(4,8,8)
# 这里不叫a了，该叫weight，因为本质上是加权平均，只是这里的加权是平均值的那种，每个位置权重一样
weight = torch.tril(torch.ones(4,8,8))
weight = weight/torch.sum(weight, dim = 2, keepdim = True)  # 关于这里的dim，看上面的示例，反正是最里面那个维度进行均一化
relation_x_teacher = weight@x
print(torch.allclose(xbow[0], relation_x_teacher[0]))

torch.Size([4, 8, 2])
True


In [46]:
%%timeit
weight = torch.tril(torch.ones(T,T))
weight = weight/torch.sum(weight, dim = 1, keepdim = True)  
relation_x_teacher = weight@x   

358 μs ± 84.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [45]:
weight = torch.tril(torch.ones(T,T))
weight = weight/torch.sum(weight, dim = 1, keepdim = True)  
relation_x_teacher = weight@x   
# (T,T)@(B,T,C) 这里会自动广播，
# 1.扩展维度为: (1,T,T); 
# 2. 广播变为: (B,T,T); 
# 3. 计算矩阵/张量乘法 (B,T,T)@(B,T,C) = (B,T,C)
# 所以会把B放在第一个维度上，就是为了方便以后的自动广播扩展
print(torch.allclose(xbow[0], relation_x_teacher[0]))

True


### 多维矩阵计算

对于多维矩阵计算，例如上面的 `(B,T,T)@(B,T,C) = (B,T,C)`

pytorch会并行且独立的对所有批次元素执行矩阵乘法运算
+ 即： 会执行`B`个 (T,T)@(T,C)的矩阵乘法，并行且独立的进行
+ 在 PyTorch 中，`@` 运算符（即 torch.matmul）支持**批量矩阵乘法（Batched Matrix Multiplication）**

**GPU 硬件并行（三维并行）**：
+ GPU 驱动将任务下发给硬件。
+ GPU 将 `B` 个批次分配给不同的计算簇（SM）。
+ 每个 SM 内部再对 `(T,T)` @ `(T,C)` 进行二维的线程级并行。
+ 最终实现：Batch维度、行维度、列维度 三个维度同时并行计算。

----
从纯数学和算法逻辑的角度来看，多维矩阵乘法（Batched Matmul）的本质，**确实是多个独立的二维矩阵乘法的集合**。
* 数学公式上：$Result_{b, i, j} = \sum_k A_{b, i, k} \times B_{b, k, j}$
* 逻辑上：它就是把一个 3D 任务，看作 $B$ 个平行的 2D 任务。
在这个层面上，说“本质上拆分成多个二维矩阵”是**完全正确**的。

---

**软件调度层面：绝对不能“拆成 B 个二维矩阵去执行”**

如果您说的“拆分”，是指在 PyTorch 或底层代码中，写一个循环，把这 $B$ 个二维矩阵**一个个单独提取出来，分 $B$ 次送给 GPU 去算**，那是**绝对错误**的。

* **为什么错？** 因为 GPU 的启动（Kernel Launch）是有延迟的。如果您拆成 $B$ 个独立的二维矩阵去执行，CPU 就要向 GPU 发送 $B$ 次指令。这会导致 GPU 大部分时间在等待 CPU 发指令，性能会暴跌几十倍甚至上百倍。
* **正确做法**：PyTorch 是把这 $B$ 个二维矩阵**打包成一个 3D 的整体任务**，一次性扔给 GPU。GPU 底层调用的 `Batched GEMM` 接口，天生就知道如何处理这个 3D 结构。

---

**硬件物理执行层面：最小的执行单位是“微小”的二维矩阵块（Tile）**

这是最核心的真相。当 GPU 接收到这个 3D 的整体任务后，它在物理层面上是怎么执行的呢？

**GPU 并不是把任务拆成 $B$ 个完整的 $(T, T)$ 二维矩阵去算，而是把整个 3D 数据“切块（Tiling）”，切成无数个极其微小的二维矩阵块！**

* **完整的二维矩阵太大了**：一个 $(T, T)$ 的矩阵（比如 $1024 \times 1024$）包含上百万个元素，GPU 的单个计算核心（CUDA Core）根本装不下，也算不了这么大一整块。
* **真正的最小物理单位（Tile/Block）**：GPU 会将数据在 Batch、Row、Column 三个维度上，切割成非常小的**二维矩阵块（通常称为 Tile，比如 $16 \times 16$ 或 $8 \times 8$ 的大小）**。
* **三维并行的真相**：GPU 拥有成千上万个计算核心。它会把这无数个微小的 $16 \times 16$ 二维矩阵块，分配给不同的计算核心。
  * 核心 A 算 Batch 0 的左上角 $16 \times 16$ 块。
  * 核心 B 算 Batch 0 的右下角 $16 \times 16$ 块。
  * 核心 C 算 Batch 1 的左上角 $16 \times 16$ 块。
  * ……

**结论**：在 GPU 硬件的最底层，执行矩阵乘法的最小物理单位，**确实是二维的（微小的二维矩阵块 Tile）**，但它**不是您原本设想的那个完整的二维矩阵**，而是被切碎后的“像素级”二维小块。并且，这些小块是在 Batch、行、列**三个维度上同时并行计算的**。

*(注：如果是使用 NVIDIA 的 Tensor Core，其最小物理指令如 `mma` 计算的是类似 $8 \times 8 \times 4$ 的微小三维块，但为了通俗理解，将其视为包含累加维度的“微小二维块”也是合理的。)*

---

多维矩阵乘法在逻辑上等价于多个二维矩阵乘法；但在物理执行时，它**不是**拆分成多个“完整的”二维矩阵去排队执行，而是将多维数据打散成无数个“微小的”二维矩阵块，进行真正的**高维空间内的全并行计算**。

### 实际上工程实现的算法

In [52]:
tril = torch.tril(torch.ones(T,T))
weight = torch.zeros((T,T))
weight = weight.masked_fill(tril==0, float('-inf'))
print(weight)
# Out-of-place version of :meth:`torch.Tensor.masked_fill_`
# 这个是 `torch.Tensor.masked_fill_` 的非原地操作版本， fill_ 是原地操作 in_place
# 这里的作用就是把下三角矩阵的右上角全部填充上 -inf
weight = F.softmax(weight, dim = -1)
relation_engineer = weight@x   

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])


其实得到的结果是类似的，
+ 对于dim=-1的第一列来说，$\frac{e^0}{e^0 + 0 + 0 + ... + 0}=1$, $\frac{e^{-inf}}{e^0 + 0 + 0 + 0 + ... + 0}=0$
+ 同理，对于dim=-1的第二列来说， $\frac{e^0}{e^0 + e^0 + 0 + ... + 0}=1$, $\frac{e^{-inf}}{e^0 + e^0  + 0 + ... + 0}=0$
+ **`softmax`本质上也是一种归一化函数**
+ 这里的改进在于，把权重变成了`e`的指数了

这个明显耗时更久一些，但是这样做的原因在于：
1. `self-attention`里用的就是这个，
2. 这里用的只是`mask+softmax`的特殊情况(weight=0)来作为关系的平均值的计算，
3. 更普遍的情况则是weight不同，用在自注意力机制中

```bash
weight = torch.zeros((T,T))  
weight = weight.masked_fill(tril==0, float('-inf')) 
# 这里把靠后的权重全部设置为负无穷大，用来表明未来的张量的权重/相关性/affinity(亲合度)为0
```

In [50]:
print(torch.allclose(xbow[0], relation_x_teacher[0]))
print(xbow[0] == relation_x_teacher[0])

True
tensor([[ True,  True],
        [ True,  True],
        [ True, False],
        [ True,  True],
        [ True, False],
        [False,  True],
        [False, False],
        [ True,  True]])


In [51]:
%%timeit
tril = torch.tril(torch.ones(T,T))
weight = torch.zeros((T,T))
weight = weight.masked_fill(tril==0, float('-inf'))
weight = F.softmax(weight, dim = -1)
relation_engineer = weight@x   

692 μs ± 30.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


### ✅总结

```bash
# 方式1， for循环计算均值
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] 
        xbow[b,t] = torch.mean(xprev, dim = 0)

# 方式2，下三角矩阵直接计算均值
weight = torch.tril(torch.ones(4,8,8))
weight = weight/torch.sum(weight, dim = 2, keepdim = True)  
xbow2 = weight@x

# 方式3，改用softmax
tril = torch.tril(torch.ones(T,T))
weight = torch.zeros((T,T))
weight = weight.masked_fill(tril==0, float('-inf'))
weight = F.softmax(weight, dim = -1)
xbow3 = weight@x   
```
+ 方式1→方式2， 从均值的计算里，for循环进化到下三角矩阵，下三角矩阵归一化
+ 方式2→方式3， 把下三角矩阵的归一化，进一步进化成了softmax进行归一化，这样得到的矩阵就刚好是权重，表示token彼此之间的相关性/affinity(亲合度)

### 对比/差距说明

矩阵乘法 vs for循环
```bash
# for循环  %%time的结果
CPU times: total: 15.6 ms
Wall time: 2.72 ms
# for循环 %%timeit的结果
3.48 ms ± 201 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

# 我的 
# 矩阵乘法 %%time的结果
CPU times: total: 0 ns
Wall time: 999 μs
# 矩阵乘法 %%timeit的结果
387 μs ± 58.9 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)

# 老师的  
# 矩阵乘法 %%timeit的结果
358 μs ± 84.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
```
+ 直接通过`%%timeit`的结果可以知道，for循环和矩阵乘法 **1 毫秒 (ms) = 1000 微秒 (μs)**
  + 基本是10倍左右的提速了
+ **Wall time（挂钟时间 / 墙上时间）**：
  + 定义：代码从开始执行到完全结束，所经历的绝对物理时间。
  + 包含内容：它不仅包含 CPU 计算的时间，还包含了所有的等待时间。比如：等待硬盘读取数据（I/O）、等待网络响应、等待获取锁、或者像 PyTorch 中等待 GPU 计算完成。
  + 意义：这是用户感知到的真实耗时，也是评估程序整体性能最核心的指标。
+ **CPU times（CPU 时间）**:
  + CPU 实际处于“工作状态”，为执行这段代码所消耗的时间。它通常由两部分组成：
  + user（用户态时间）：CPU 执行你写的 Python 代码，以及底层 C/C++ 库（如 NumPy, PyTorch 的 CPU 算子）所花费的时间。
  + sys（系统态时间）：CPU 执行操作系统内核代码所花费的时间。比如你的代码请求分配内存、读写文件、进行网络通信时，操作系统内核介入处理的时间。
  + total：user + sys 的总和。

另外，从下面可以看出，如果是`allclose`, 则为true；如果是绝对的二进制表示等于，则有false;

这是因为，矩阵乘法本身执行的是加权平均，而不像for循环那样直接`mean`

```python
# for循环
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]  # 得到的结果是 (t, C)
        xbow[b,t] = torch.mean(xprev, dim = 0)  # 只有这里会引入误差

# 矩阵乘法
weight = torch.tril(torch.ones(T,T))
weight = weight/torch.sum(weight, dim = 1, keepdim = True)  # 第一次引入误差
relation_x_teacher = weight@x     # 第二次引入误差， 这里引入的误差其实和 torch.mean(xprev, dim = 0)  差不多

# 所以其实for循环的误差更小一点
```

In [48]:
print(torch.allclose(xbow[0], relation_x_teacher[0]))
print(xbow[0] == relation_x_teacher[0])

True
tensor([[ True,  True],
        [ True,  True],
        [ True, False],
        [ True,  True],
        [ True, False],
        [False,  True],
        [False, False],
        [ True,  True]])


# 其他

## nn.Embedding层随机初始化

嵌入层的初始化， 根据`3_MLP_makemore.ipynb`

```python
C = torch.randn((27,2))
emb = C[X]
print(f"emb.shape: {emb.shape}")
```

根据[Reference API -> torch.nn -> Embedding](https://docs.pytorch.org/docs/2.13/generated/torch.nn.Embedding.html)
```bash
Variables: weight (Tensor) – the learnable weights of the module of shape (num_embeddings, embedding_dim) initialized from N(0,1)

# 只写：nn.Embedding(num_embeddings, embedding_dim)
# PyTorch 会创建一个形状为：(num_embeddings, embedding_dim)的可训练参数 weight，并且默认会用N(mean=0, std=1)初始化
```
根据[torch/nn/modules/sparse.py](https://github.com/pytorch/pytorch/blob/v2.13.0/torch/nn/modules/sparse.py#L164-L181)
```python
if _weight is None:
    self.weight = Parameter(
        torch.empty((num_embeddings, embedding_dim), **factory_kwargs),
        requires_grad=not _freeze,
    )
    self.reset_parameters()   # 当没有对nn.Embedding加载外部训练好的权重的时候，会先创建一个空tensor(即：self.weight)，然后调用这个函数对空tensor赋值
...

def reset_parameters(self) -> None:
    init.normal_(self.weight)   # 这里就是对 self.weight真正进行初始化的地方
    self._fill_padding_idx_with_zero()

from torch.nn import functional as F, init
# 再结合  https://docs.pytorch.org/docs/2.13/nn.init.html#torch.nn.init.normal_
torch.nn.init.normal_(tensor, mean=0.0, std=1.0, generator=None)[source]
# Fill the input Tensor with values drawn from the normal distribution.
```
可知，  embedding table 中的每个元素默认来自标准正态分布，不存在以行为单位进行正态分布的采样这个说法

每个标量元素独立地从标准正态分布采样。所以，如果你说的“以行为单位进行正态分布采样”是指：
```bash
每一行有一个单独的均值、方差，或者每一行调用一次类似 normal_(dim=1) 的采样接口
```
那默认实现里没有这种说法，也没有 `dim=1` 这种参数。

## optimizer.zero_grad(set_to_none = True)

根据 [torch.optim.Optimizer.zero_grad](https://docs.pytorch.org/docs/2.13/generated/torch.optim.Optimizer.zero_grad.html)

```bash
不要将梯度设为零，而是将其设为 None。默认值：True。

这通常会减少内存占用，并能适度提升性能。但会改变某些行为。例如：

当用户尝试访问梯度并对其进行手动操作时，None属性或充满0的张量会表现出不同的行为。

如果用户先请求 zero_grad(set_to_none=True)，然后进行反向传播，对于未接收梯度的参数，.grads 保证为 None。

torch.optim 优化器在梯度为 0 或 None 时具有不同的行为（一种情况下使用梯度为 0 的步长，另一种情况下则完全跳过该步）。
```